# 08 — Correct & Rebuild the Manuscript Evidence Table

**Purpose.** Take the *clean* reviewed outputs from notebooks 06/07 and fix the faulty
findings found in spot-checking, then **overwrite the existing files in place** (same
filenames — no dated copies).

**Integrity rules (important).**
- No numeric value is invented. Every value that survives already existed in the source extraction.
- Three correction actions only:
  - `drop_duplicate` — exact repeats of a kept row.
  - `quarantine` — misread / junk / placeholder values removed from the citable table
    and parked in a `Quarantine_Rejected` sheet with a reason + re-extraction note.
  - `reclassify` — a *real* source figure that was filed under the wrong metric/role
    (e.g. a % growth in construction *value* labelled as a causal jobs effect) is relabelled
    and moved to the Scale-Context sheet.
- Every surviving row still carries `spotcheck_required` — the workbook is a corrected
  scaffold, not yet source-verified.

**What this notebook does NOT do.** No new web search, downloads, or re-extraction. The
two high-priority items that need fresh extraction (real Alvarez causal coefficients; a
broader comparator base) are logged in the Gap sheet, not faked.


In [ ]:
import os, json, shutil, datetime
import pandas as pd
import numpy as np

# ---------------- Config ----------------
OVERWRITE_IN_PLACE = True      # overwrite existing outputs (no dated files)
MAKE_BACKUP        = True      # ONE fixed-name backup before overwriting (not dated)
BACKUP_DIRNAME     = "_pre_correction_backup"

# Files this notebook owns and overwrites in place (fixed names, never dated)
OUT_FILES = [
    "manuscript_primary_evidence_table.xlsx",
    "table_1_data_centre_vs_comparators.csv",
    "figure_job_intensity_clean_data.csv",
    "scale_context_for_text.csv",
    "source_spotcheck_queue.csv",
]

# ------- Resolve manuscript_tables_figures dir (works locally or in the repo) -------
CANDIDATES = [
    # Primary location (after consolidation into Manuscript/):
    "../../../Manuscript/evidence/01_primary_evidence_tables",
    "DC_job_potential/Manuscript/evidence/01_primary_evidence_tables",
    "LOCAL_WORKSPACE_ROOT/"
    "LEADERSHIP/AI for Construction/PUBLISH/DC_job_potential/Manuscript/"
    "evidence/01_primary_evidence_tables",
    # Legacy location (pre-consolidation), kept as fallback:
    "../../03_Synthesis/manuscript_tables_figures",
    "LOCAL_WORKSPACE_ROOT/"
    "LEADERSHIP/AI for Construction/PUBLISH/DC_job_potential/Research_Workflow/"
    "03_Synthesis/manuscript_tables_figures",
]
MS_DIR = next((p for p in CANDIDATES
               if os.path.isfile(os.path.join(p, "manuscript_primary_evidence_table.xlsx"))), None)
assert MS_DIR, "Could not locate manuscript_tables_figures dir. Set MS_DIR manually."
MS_DIR = os.path.abspath(MS_DIR)
XLSX = os.path.join(MS_DIR, "manuscript_primary_evidence_table.xlsx")
print("Working dir:", MS_DIR)


In [ ]:
# ---------------- Load the PRISTINE source, then back it up ----------------
# To stay idempotent, every run corrects from the pre-correction snapshot:
#   - first run  : read the live file, then snapshot it into the backup folder
#   - later runs : read from the backup snapshot (live files are already corrected)
bdir = os.path.join(MS_DIR, BACKUP_DIRNAME)
os.makedirs(bdir, exist_ok=True)
backup_xlsx = os.path.join(bdir, "manuscript_primary_evidence_table.xlsx")

if os.path.isfile(backup_xlsx):
    SRC = backup_xlsx
    print("Reading pristine source from backup (idempotent re-run):", SRC)
else:
    SRC = XLSX
    if MAKE_BACKUP:                       # snapshot the pristine originals once
        for f in OUT_FILES:
            s = os.path.join(MS_DIR, f)
            if os.path.isfile(s):
                shutil.copy2(s, os.path.join(bdir, f))
        print("Created pristine backup at:", bdir)

xls = pd.ExcelFile(SRC)
print("sheets found:", xls.sheet_names)
sheets = {name: xls.parse(name, dtype=object) for name in xls.sheet_names}

t1     = sheets["Table1_Evidence"].copy()
fig    = sheets["FigureData_Clean"].copy()
scx    = sheets["ScaleContext_Text"].copy()
spq    = sheets.get("SpotcheckQueue")
capn   = sheets.get("CaptionNotes")
gap    = sheets.get("GapLog_From06")

print(f"Table1 rows: {len(t1)} | Figure rows: {len(fig)} | ScaleContext rows: {len(scx)}")
assert "evidence_id" in t1.columns, "Expected an evidence_id column in Table1_Evidence."


## Corrections register

Keyed on the stable `evidence_id`. Edit any decision here and re-run — every change flows
through to the workbook, the CSVs, the figure dataset and the audit sheets.


In [ ]:
# ---------------- Corrections register ----------------

# dropped_evidence_id : kept_evidence_id  (exact repeats)
DUPLICATES = {
    "ev_0011": "ev_0004",  # 2.5 jobs/MW, Meta Richland Parish, LA  (repeat)
    "ev_0010": "ev_0005",  # 3.6 jobs/MW, Vantage Frontier, TX      (repeat)
    "ev_0009": "ev_0006",  # 1.3 jobs/MW, OpenAI Stargate Abilene, TX (repeat)
    "ev_0019": "ev_0018",  # 50 operational FTE, Yue & Zeng         (repeat)
    "ev_0003": "ev_0001",  # 0.7-2.0 jobs/MW, Hamm (mislabelled as construction_jobs_headcount; dup of jobs_per_mw row)
    "ev_0049": "ev_0020",  # 1,500 Virginia (mislabelled as construction_duration; dup of headcount row)
}

# evidence_id : reason (misread / junk / placeholder -> removed from citable table)
QUARANTINE = {
    "ev_0043": "Alvarez NBER Table 1 'Summary statistics': 0.858 is the Std. dev. of %\u0394 employment "
               "(Obs 3,173, Mean 0.212), NOT a causal effect. Re-extract the treatment-effect/DiD coefficients.",
    "ev_0044": "Alvarez NBER summary-stat Std. dev. of %\u0394 data-processing employment (1.627); not a causal effect.",
    "ev_0045": "Alvarez NBER summary-stat Std. dev. of %\u0394 construction employment (1.530); not a causal effect.",
    "ev_0046": "Alvarez NBER summary-stat Std. dev. (1.039); not a causal effect.",
    "ev_0052": "Yue & Zeng: '5' is a fragment from the CS-DID heterogeneity table header, not a clean effect size.",
    "ev_0060": "NSW submission: '4' relates to apprenticeship/licensing text, not investment scale. Mis-extracted.",
    "ev_0061": "NSW submission: '3' = 'past 3 years' (DC investment grew ~65%/yr; ~12% of non-residential). "
               "Value '3' mis-extracted; recover the 65%/12% figures on re-extraction.",
    "ev_0073": "Gargano & Giacoletti: '150' is a fragment from the tax-incentive legislation table; not an employment effect.",
    "ev_0074": "Gargano & Giacoletti: '120' is a fragment from the tax-incentive legislation table; not an employment effect.",
    "ev_0093": "Comparator (Putting renewables to work): explicit placeholder (0.15 job-years/GWh) and a duplicate of ev_0091; not citable.",
}

# evidence_id : corrected fields (real figure, wrong bin -> relabel + move to Scale Context)
RECLASSIFY = {
    "ev_0068": dict(metric_family="construction_work_value_growth_pct",
                    verified_value="246% increase in value of data-centre construction work done since Mar 2022 (ABS; Sep 2025 ~$1.03b)",
                    evidence_role="scale_context_only",
                    manuscript_use="Market-growth / scale context only - NOT an employment or causal-jobs effect",
                    caveat_summary="Growth in construction work VALUE, not jobs. Do not present as an employment effect."),
    "ev_0071": dict(metric_family="construction_cost_cagr_pct",
                    verified_value="7% CAGR in average global DC construction cost per MW, 2020-2025 ($7.7M -> $10.7M/MW)",
                    evidence_role="scale_context_only",
                    manuscript_use="Cost-trend / scale context only - NOT an employment or causal-jobs effect",
                    caveat_summary="Cost CAGR, not jobs. Do not present as an employment effect."),
}

print(f"duplicates={len(DUPLICATES)}  quarantine={len(QUARANTINE)}  reclassify={len(RECLASSIFY)}")


In [ ]:
# ---------------- Apply corrections ----------------
EVID = "evidence_id"
dup_ids  = set(DUPLICATES)
quar_ids = set(QUARANTINE)
recl_ids = set(RECLASSIFY)
removed_ids = dup_ids | quar_ids          # leave the citable table entirely

# audit log
rows = []
for eid, keep in DUPLICATES.items():
    rows.append(dict(evidence_id=eid, action="drop_duplicate", kept_evidence_id=keep,
                     reason=f"Exact duplicate of {keep}."))
for eid, why in QUARANTINE.items():
    rows.append(dict(evidence_id=eid, action="quarantine", kept_evidence_id="", reason=why))
for eid, f in RECLASSIFY.items():
    rows.append(dict(evidence_id=eid, action="reclassify_to_scale_context", kept_evidence_id="",
                     reason=f["caveat_summary"]))
corrections_log = pd.DataFrame(rows)

# snapshots taken from the ORIGINAL table
quarantine_rows = t1[t1[EVID].isin(quar_ids)].merge(
    corrections_log[["evidence_id", "reason"]], on="evidence_id", how="left")
duplicates_removed = t1[t1[EVID].isin(dup_ids)].copy()
duplicates_removed["kept_evidence_id"] = duplicates_removed[EVID].map(DUPLICATES)

# reclassified rows (relabelled)
reclassified = t1[t1[EVID].isin(recl_ids)].copy()
for eid, fdict in RECLASSIFY.items():
    for col, val in fdict.items():
        reclassified.loc[reclassified[EVID] == eid, col] = val

# corrected, citable Table 1
drop_all = removed_ids | recl_ids
t1_corr = t1[~t1[EVID].isin(drop_all)].copy().reset_index(drop=True)
if "table_row_id" in t1_corr.columns:
    t1_corr["table_row_id"] = [f"t1_{i+1:03d}" for i in range(len(t1_corr))]

print(f"Original Table1: {len(t1)}  ->  corrected citable: {len(t1_corr)}")
print(f"  removed duplicates : {len(duplicates_removed)}")
print(f"  quarantined        : {len(quarantine_rows)}")
print(f"  reclassified->scale: {len(reclassified)}")


In [ ]:
# ---------------- Scale-context: append the reclassified rows ----------------
scx_corr = scx.copy()
add = []
for _, r in reclassified.iterrows():
    row = {c: r[c] for c in scx_corr.columns if c in r.index}
    row["evidence_id"] = r["evidence_id"]
    for c in ["metric_family", "verified_value", "evidence_role"]:
        if c in scx_corr.columns:
            row[c] = r.get(c)
    if "text_use" in scx_corr.columns:
        row["text_use"] = "Scale / market context only - NOT an employment effect"
    if "notes" in scx_corr.columns:
        row["notes"] = "RECLASSIFIED from Table 1 (was mislabelled as a causal/jobs effect). " + str(r.get("caveat_summary", ""))
    add.append(row)
if add:
    scx_corr = pd.concat([scx_corr, pd.DataFrame(add)], ignore_index=True)
if "scale_row_id" in scx_corr.columns:
    scx_corr["scale_row_id"] = [f"sc_{i+1:03d}" for i in range(len(scx_corr))]

# ---------------- Figure dataset: exclude rows built on removed/reclassified evidence ----------------
fig_corr = fig.copy()
SEI = "source_evidence_id"
def _ids(v):
    return [x.strip() for x in str(v).split(";")] if pd.notna(v) else []

# A figure row is excluded if it touches genuinely bad evidence
# (quarantined misread/junk/placeholder, or reclassified-out-of-jobs).
# Duplicate-collapse rows reference BOTH the kept row and its dropped twin, so they
# are kept as long as a surviving (non-dropped) evidence id remains.
hard_bad = quar_ids | recl_ids
def _fig_excluded(v):
    ids = _ids(v)
    if any(i in hard_bad for i in ids):
        return True
    surviving = [i for i in ids if i not in dup_ids]
    return len(surviving) == 0
mask_bad = fig_corr[SEI].apply(_fig_excluded)

if "plot_ready" in fig_corr.columns:
    fig_corr.loc[mask_bad, "plot_ready"] = False
if "figure_use" in fig_corr.columns:
    fig_corr.loc[mask_bad, "figure_use"] = "excluded_after_correction"
if "notes" in fig_corr.columns:
    fig_corr.loc[mask_bad, "notes"] = ("EXCLUDED in correction pass (evidence quarantined/reclassified/duplicate). "
                                       + fig_corr.loc[mask_bad, "notes"].fillna("").astype(str))

n_ready = int((fig_corr["plot_ready"] == True).sum()) if "plot_ready" in fig_corr.columns else None
print(f"Figure rows flagged excluded: {int(mask_bad.sum())}")
print(f"Figure rows still plot_ready=True: {n_ready}")


In [ ]:
# ---------------- Rebuild SpotcheckQueue from corrected citable rows ----------------
if spq is not None:
    base_cols = [c for c in spq.columns if c in t1_corr.columns]
    spq_corr = t1_corr[base_cols].copy()
    if "spotcheck_id" in spq.columns:
        spq_corr.insert(0, "spotcheck_id", [f"sc_chk_{i+1:03d}" for i in range(len(spq_corr))])
    if "spotcheck_priority" in spq.columns and "spotcheck_priority" not in spq_corr.columns:
        spq_corr["spotcheck_priority"] = "high"
    if "spotcheck_instruction" in spq.columns and "spotcheck_instruction" not in spq_corr.columns:
        spq_corr["spotcheck_instruction"] = "Verify value, unit, scope, and page against the source PDF before citing."
else:
    spq_corr = t1_corr.copy()

# ---------------- Update Gap log ----------------
gap_corr = gap.copy() if gap is not None else pd.DataFrame(
    columns=["gap_id", "gap_type", "priority", "issue", "recommended_action"])
new_gaps = [
    dict(gap_id="gap_006", gap_type="extraction_error_fixed", priority="high",
         issue="Alvarez NBER 'causal' rows were summary-stat Std. devs (0.858/1.627/1.530/1.039); quarantined this pass.",
         recommended_action="Re-extract Alvarez treatment-effect / DiD coefficients from the results tables, then re-admit."),
    dict(gap_id="gap_007", gap_type="extraction_error_fixed", priority="high",
         issue="Non-employment figures (ABS 246% work value; JLL 7% cost CAGR; Gargano tax-table 150/120; NSW 4/3) were filed as causal/investment effects.",
         recommended_action="ABS & JLL reclassified to scale-context; Gargano & NSW values quarantined as mis-extracted."),
    dict(gap_id="gap_008", gap_type="comparator_coverage", priority="high",
         issue="All comparator rows derive from 2 renewable-energy reports; 'data centre vs comparator assets' is currently 'vs renewable energy'.",
         recommended_action="Targeted gap search for high-rise/commercial, advanced-manufacturing, transport-infrastructure job-intensity metrics, OR narrow the comparison claim explicitly."),
]
gap_corr = pd.concat([gap_corr, pd.DataFrame(new_gaps)], ignore_index=True)

# ---------------- README ----------------
dc = (t1_corr["domain"] == "Data centre").sum() if "domain" in t1_corr.columns else None
cmp = (t1_corr["domain"] == "Comparator asset").sum() if "domain" in t1_corr.columns else None
readme_corr = pd.DataFrame([
    ("table_title", "Manuscript primary evidence table (corrected)"),
    ("corrected_on", datetime.date.today().isoformat()),
    ("basis", "Notebook 08: corrections applied to notebook 06/07 reviewed outputs. No new search/extraction; no values invented."),
    ("table_1_rows_citable", len(t1_corr)),
    ("  of which data_centre", dc),
    ("  of which comparator", cmp),
    ("duplicates_removed", len(duplicates_removed)),
    ("quarantined_misread_or_placeholder", len(quarantine_rows)),
    ("reclassified_to_scale_context", len(reclassified)),
    ("figure_rows_total", len(fig_corr)),
    ("figure_rows_plot_ready", n_ready),
    ("scale_context_rows", len(scx_corr)),
    ("spotcheck_rows", len(spq_corr)),
    ("main_warning", "Every row still requires a source/PDF spot-check before citation. "
                     "Keep direct_vs_total, projected_vs_realised, evidence_role and caveats visible. Do not average unlike units."),
], columns=["item", "value"])
print(readme_corr.to_string(index=False))


In [ ]:
# ---------------- Integrity checks (run BEFORE writing) ----------------
orig_vals = set(t1[EVID])
assert set(t1_corr[EVID]).issubset(orig_vals), "A new evidence_id appeared - nothing should be invented."
accounted = set(t1_corr[EVID]) | removed_ids | recl_ids
assert accounted == orig_vals, ("Unaccounted original rows:", orig_vals - accounted)
if "plot_ready" in fig_corr.columns:
    still_bad = fig_corr[(fig_corr["plot_ready"] == True) &
                         (fig_corr[SEI].apply(lambda v: any(i in (quar_ids | recl_ids) for i in _ids(v))))]
    assert len(still_bad) == 0, f"{len(still_bad)} figure rows still plot_ready on quarantined/reclassified evidence"
print("All integrity checks passed.")


In [ ]:
# ---------------- Write outputs (OVERWRITE in place, fixed names) ----------------
# (Pristine backup was already taken at load time, before any overwrite.)
assert OVERWRITE_IN_PLACE, "Set OVERWRITE_IN_PLACE=True to write."

capn_out = capn if capn is not None else pd.DataFrame()
with pd.ExcelWriter(XLSX, engine="openpyxl") as xw:           # overwrite same .xlsx
    readme_corr.to_excel(xw, sheet_name="README", index=False)
    t1_corr.to_excel(xw, sheet_name="Table1_Evidence", index=False)
    fig_corr.to_excel(xw, sheet_name="FigureData_Clean", index=False)
    scx_corr.to_excel(xw, sheet_name="ScaleContext_Text", index=False)
    spq_corr.to_excel(xw, sheet_name="SpotcheckQueue", index=False)
    capn_out.to_excel(xw, sheet_name="CaptionNotes", index=False)
    gap_corr.to_excel(xw, sheet_name="GapLog_From06", index=False)
    corrections_log.to_excel(xw, sheet_name="Corrections_Log", index=False)
    quarantine_rows.to_excel(xw, sheet_name="Quarantine_Rejected", index=False)
    duplicates_removed.to_excel(xw, sheet_name="Duplicates_Removed", index=False)

# overwrite companion CSVs (same filenames)
t1_corr.to_csv(os.path.join(MS_DIR, "table_1_data_centre_vs_comparators.csv"), index=False)
fig_corr.to_csv(os.path.join(MS_DIR, "figure_job_intensity_clean_data.csv"), index=False)
scx_corr.to_csv(os.path.join(MS_DIR, "scale_context_for_text.csv"), index=False)
spq_corr.to_csv(os.path.join(MS_DIR, "source_spotcheck_queue.csv"), index=False)
# fixed-name audit CSVs alongside
corrections_log.to_csv(os.path.join(MS_DIR, "corrections_log.csv"), index=False)
quarantine_rows.to_csv(os.path.join(MS_DIR, "quarantine_rejected.csv"), index=False)

print("Overwrote in place:")
for f in OUT_FILES + ["corrections_log.csv", "quarantine_rejected.csv"]:
    print("  -", f)
print(f"\nCorrected citable Table 1: {len(t1_corr)} rows  (data-centre={dc}, comparator={cmp})")


## Still required before citing (logged, not done here)

1. **Run the spot-check queue** — every surviving row is still `spotcheck_required`.
2. **Re-extract the Alvarez causal effects** (the quarantined rows used dispersion stats).
   Pull the actual DiD / treatment-effect coefficients, then re-admit to Table 1.
3. **Broaden the comparator base** beyond the two renewable-energy reports, *or* narrow the
   manuscript's comparison claim to "energy assets" and state it plainly.

Re-running this notebook is safe and idempotent: it always rebuilds from `Table1_Evidence`,
re-applies the register, and overwrites the same files. To restore the pre-correction state,
copy the files back from the `_pre_correction_backup/` folder.
